Data Science Capstone - Week 2
-
<h3> Linear Regression </h3>
Keshia-Lee Martin

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [11]:
movie_box_office = pd.read_csv("movie_box_office_cv2.csv")

# dropping titles that don't have genre, country, language, votes listed
movie_box_office = movie_box_office[movie_box_office['Vote_Count'].notna()].copy()
print('Rows Remaining:', len(movie_box_office))

# target as the random forest: log10 of the worldwide revenue
movie_box_office['Log_Worldwide'] = np.log10(movie_box_office['$Worldwide'])
movie_box_office['Log_Worldwide'].describe().round(3)

Rows Remaining: 4830


count    4830.000
mean        7.758
std         0.514
min         6.222
25%         7.399
50%         7.695
75%         8.090
max         9.447
Name: Log_Worldwide, dtype: float64

Pre-release features, excluding Rating and Vote_Count, which is post-release.\
Rank and the domestic and foreign revenue columns, because they are pieces of the target.

In [12]:
# Feature names have no spaces, so interaction names read cleaner for later
movie_box_office = movie_box_office.rename(columns={'Science Fiction': 'Sci_Fi', 'TV Movie': 'TV_Movie'})
genre_cols = ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama',
              'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance',
              'Sci_Fi', 'TV_Movie', 'Thriller', 'War', 'Western']

In [13]:
# take all rows and only the columns whose labels fall in this range for country names
country_cols = list(movie_box_office.loc[:, 'Algeria':'Yugoslavia'].columns)

In [14]:
# Release years and also the pandemic years
movie_box_office['Years_Since_2000'] = movie_box_office['Year'] - 2000
movie_box_office['Pandemic_Release'] = movie_box_office['Year'].isin([2020, 2021]).astype(int)

In [17]:
# The 10 most common countries, plus how many countries are on the film
top_countries = {'United States of America': 'Prod_USA', 'United Kingdom': 'Prod_UK',
                 'France': 'Prod_France', 'Japan': 'Prod_Japan', 'China': 'Prod_China',
                 'Germany': 'Prod_Germany', 'Canada': 'Prod_Canada', 'South Korea': 'Prod_South_Korea',
                 'Hong Kong': 'Prod_Hong_Kong', 'India': 'Prod_India'}

# creating new columns from the top countries
production_cols = []
for country in top_countries:
    col_name = top_countries[country]
    movie_box_office[col_name] = movie_box_office[country]
    production_cols.append(col_name)
movie_box_office['Num_Countries'] = movie_box_office[country_cols].sum(axis=1)

In [19]:
# Original language: English is the baseline, so it gets no column
top_languages = {'Japanese': 'Lang_Japanese', 'Chinese (Mandarin)': 'Lang_Mandarin',
                 'French': 'Lang_French', 'Korean': 'Lang_Korean', 'Hindi': 'Lang_Hindi',
                 'German': 'Lang_German', 'Spanish': 'Lang_Spanish',
                 'Chinese (Cantonese)': 'Lang_Cantonese', 'Italian': 'Lang_Italian'}

language_cols = []
for language in top_languages:
    col_name = top_languages[language]
    movie_box_office[col_name] = (movie_box_office['Original_Language'] == language).astype(int)
    language_cols.append(col_name)
named_languages = ['English'] + list(top_languages.keys())
movie_box_office['Lang_Other'] = (~movie_box_office['Original_Language'].isin(named_languages)).astype(int)
language_cols.append('Lang_Other')

In [21]:
feature_cols = ['Years_Since_2000', 'Pandemic_Release', 'Num_Countries'] + genre_cols + language_cols + production_cols
X = movie_box_office[feature_cols]
y = movie_box_office['Log_Worldwide']
print('Number of pre-release features:', len(feature_cols))

Number of pre-release features: 42
